# Phase 2 — FINAL_TEST: unchanged Qwen3-4B + the frozen adapters (PREREGISTRATION §11 with Amendments 1–3)
**Run once.** 1 + 3S checkpoints × 100 families × E/R/C (3,000 generations at S = 3; it resumes across sessions from Drive: re-run all; generated responses are never regenerated or retried).
The gate below refuses to run unless (1) the embedded `phase2/PREREGISTRATION.md` (with Amendments 1 and 2) has SHA-256 `a22c81fb33b4d3339770a2b28c51d8d3706af2b5dd038d46aa89fbb56139714d`, (2) the embedded freeze manifest `phase2/ADAPTERS_FROZEN.md` is intact, records the embedded implementation manifest, the unchanged-DEV baseline and at least one complete seed triplet, (3) every retained adapter on Drive (`MyDrive/Hinglish_Phase2/train/`) is a valid run (60/60 successful steps, not resumed), paired within its seed, hash-identical to the manifest, and the attempt log still yields the frozen seeds, every attempt having been started after the frozen unchanged-DEV baseline under the frozen implementation, and (4) that baseline's results ZIP on Drive (`MyDrive/Hinglish_Phase2/baseline_dev/`) still has its frozen SHA-256 (Amendment 2 item 6). The GPU cell refuses any stack other than the exact §2 stack on a T4 (Amendment 2 item 3). S = 3 is confirmatory; S = 1 or 2 runs only as a **descriptive-only reduced-S report**; S = 0 never reaches this notebook.
**Sealed (Amendment 1 item 14):** this notebook prints only completion, checksum and infrastructure status. Answers, scores and accuracy are never shown; the scored records stay in the results ZIP until every evaluation is complete. Frozen protocol (PREREGISTRATION §4): prompt `test1-english-final-v2`, greedy non-thinking decoding, NF4 + FP16, SDPA math, output cap 1,024 tokens, scorer v1.0.0, engine seed 11. **Do not change any setting.** Adapters are loaded unmerged on the same NF4 base. The analysis script runs afterwards, unchanged.
**Stale lock (Amendment 2 item 5).** If a runtime was killed during an evaluation, re-running may stop with "Run lock exists". First make sure no other Colab session is running this notebook, then set `RECOVER_STALE_LOCK = True` and re-run all: the lock is removed, a timestamped record is written next to the run, and generation resumes from the stored journal (nothing is regenerated). Set it back to False afterwards.
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. A results ZIP with checksums downloads at the end; send it unchanged.


In [ ]:
import os, sys, json, base64, io, zipfile, hashlib, subprocess, time
from pathlib import Path
USE_DRIVE = True  # the adapters and the unchanged-DEV baseline are on Drive; required
RECOVER_STALE_LOCK = False  # Amendment 2 item 5: set True ONLY after confirming that no other Colab session is running this evaluation (a killed runtime left RUNNING.lock); the recovery is logged with a timestamp and stored outputs are never regenerated
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'  # download bars would become thousands of streamed lines
PACKAGE = Path('/content/hinglish_phase2') if Path('/content').exists() else Path.cwd() / 'hinglish_phase2'
PACKAGE.mkdir(parents=True, exist_ok=True)
blob = base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest() == '1b8e780a5d6a97cfa2f9f8ae0cb64ba1fa37060570c6f09323ca4767e5b2990c'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(PACKAGE)
INPUTS = PACKAGE / 'inputs'
# Implementation manifest (PREREGISTRATION Amendment 1 item 14): refuse to run if the embedded code differs from it.
MANIFEST_TEXT = base64.b64decode('...').decode('utf-8')
MANIFEST_SHA256 = '726241c5726aaba2c51fc027866dd572abfc13d02ef9235d1901af87a9f5b43d'
assert hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest() == MANIFEST_SHA256, 'Embedded IMPLEMENTATION_MANIFEST.md was changed: refusing to run'
MANIFEST = json.loads(MANIFEST_TEXT.split('```json\n', 1)[1].split('\n```', 1)[0])
assert MANIFEST['preregistration']['sha256'] == 'a22c81fb33b4d3339770a2b28c51d8d3706af2b5dd038d46aa89fbb56139714d', 'Manifest is for another PREREGISTRATION.md: refusing to run'
# FinalTest payload: every code member is checked below; the gate checks ADAPTERS_FROZEN.md.
for member, source in {'hinglish_pilot/__init__.py': 'test1_pilot/hinglish_pilot/__init__.py', 'hinglish_pilot/data.py': 'test1_pilot/hinglish_pilot/data.py', 'hinglish_pilot/engine.py': 'test1_pilot/hinglish_pilot/engine.py', 'hinglish_pilot/hf_backend.py': 'test1_pilot/hinglish_pilot/hf_backend.py', 'hinglish_pilot/report.py': 'test1_pilot/hinglish_pilot/report.py', 'hinglish_pilot/scorer_v1.py': 'test1_pilot/hinglish_pilot/scorer_v1.py', 'secondary_analysis/__init__.py': 'secondary_analysis/__init__.py', 'secondary_analysis/cmi.py': 'secondary_analysis/cmi.py', 'secondary_analysis/numeric_endpoint.py': 'secondary_analysis/numeric_endpoint.py', 'secondary_analysis/report_secondary.py': 'secondary_analysis/report_secondary.py', 't4_smoke/train_smoke.py': 't4_smoke/train_smoke.py', 't4_smoke/labels.py': 't4_smoke/labels.py', 't4_smoke/protocol.py': 't4_smoke/protocol.py', 't4_smoke/train_phase2.py': 't4_smoke/train_phase2.py', 't4_smoke/phase2_eval.py': 't4_smoke/phase2_eval.py'}.items():
    assert hashlib.sha256((PACKAGE / member).read_bytes()).hexdigest() == MANIFEST['files'][source], f'{member} differs from the implementation manifest: refusing to run'
(INPUTS / 'IMPLEMENTATION_MANIFEST.md').write_text(MANIFEST_TEXT, encoding='utf-8')
print('Implementation manifest verified:', MANIFEST_SHA256)
if USE_DRIVE and Path('/content').exists():  # Drive only inside Colab
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = Path('/content/drive/MyDrive/Hinglish_Phase2')
else:
    STORE = PACKAGE / 'runs'
STORE.mkdir(parents=True, exist_ok=True)
try:  # Qwen3-4B is ungated; a token only avoids anonymous rate limits.
    from google.colab import userdata
    os.environ['HF_TOKEN'] = (userdata.get('HF_TOKEN') or '').strip() or os.environ.get('HF_TOKEN', '')
except Exception as error:
    print('No HF_TOKEN Colab secret (' + type(error).__name__ + '); downloading anonymously.')
if not os.environ.get('HF_TOKEN'):
    os.environ.pop('HF_TOKEN', None)

def step(log_dir, script, *args):
    """Run one phase in a fresh Python process, stream its output and log the attempt."""
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', str(PACKAGE / 't4_smoke' / script), *map(str, args)]
    started, returncode = time.time(), None
    try:
        with open(log_dir / 'log.txt', 'a') as log, subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                                     text=True, errors='replace') as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
            except BaseException:  # interrupt: never leave a phase running on the GPU behind the notebook
                process.kill()
                process.wait()
                log.write('[notebook] interrupted; phase process killed\n')
                raise
        returncode = process.returncode
    finally:
        with open(log_dir / 'attempts.jsonl', 'a') as attempts:
            attempts.write(json.dumps({'script': script, 'args': list(map(str, args)), 'returncode': returncode,
                                       'started': started, 'seconds': time.time() - started}) + '\n')
    return returncode

def zip_results(folder, archive, prefix):
    """All small files under ``folder``; weights, optimizer state and per-attempt journals stay out (hashes listed)."""
    files, large = [], {}
    for path in sorted(Path(folder).rglob('*')):
        if not path.is_file() or 'journal' in path.relative_to(folder).parts or path.name.endswith('.lock'):
            continue
        if path.suffix in ('.pt', '.safetensors'):
            large[str(path.relative_to(folder))] = hashlib.sha256(path.read_bytes()).hexdigest()
        else:
            files.append(path)
    sums = ''.join(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(folder)}\n' for p in files)
    sums += ''.join(f'{digest}  {name}  (not included)\n' for name, digest in large.items())
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in files:
            z.write(path, str(Path(prefix) / path.relative_to(folder)))
        z.writestr(str(Path(prefix) / 'SHA256SUMS.txt'), sums)
    print(sums)
    print('Return:', archive, 'SHA-256', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
    try:
        from google.colab import files as colab_files
        colab_files.download(str(archive))
    except ImportError:
        print('Local: ZIP saved on disk.')


In [ ]:
# GATE (model-free): refuse before any model is loaded.
sys.path.insert(0, str(PACKAGE / 't4_smoke'))
import phase2_eval
GATE_PASSED = False
PREREG_SHA256 = 'a22c81fb33b4d3339770a2b28c51d8d3706af2b5dd038d46aa89fbb56139714d'
FROZEN_SHA256 = '1bebf10d708620b2b8f5c11aac340ca76b12e4c990dbfff9dcc82a8f9f20929f'
TRAIN_DATA_SHA256 = 'db1bee2ce2b41c12ae8ed600355a24d6dc8985ee1181c1e86e22ffc723a467fc'  # the 160 frozen TRAIN families every adapter must have been trained on
phase2_eval.check_preregistration((INPUTS / 'PREREGISTRATION.md').read_bytes(), PREREG_SHA256)
frozen_text = (INPUTS / 'ADAPTERS_FROZEN.md').read_text(encoding='utf-8')
assert hashlib.sha256(frozen_text.encode('utf-8')).hexdigest() == FROZEN_SHA256, 'Embedded freeze manifest changed: FINAL_TEST refused'
FROZEN = phase2_eval.parse_frozen(frozen_text)
assert FROZEN['implementation_manifest_sha256'] == MANIFEST_SHA256, 'Adapters were frozen under another implementation manifest: FINAL_TEST refused'
S = FROZEN['S']
BASELINE_DIR = STORE / 'baseline_dev'  # Amendment 2 item 6: the frozen unchanged-DEV run
CHECKPOINTS = [('base', None, None)] + [(phase2_eval.run_name(a['arm'], a['seed']), a, folder)
                                         for a, folder in phase2_eval.gate(FROZEN, STORE / 'train', expected_data_sha256=TRAIN_DATA_SHA256,
                                                                           baseline_dir=BASELINE_DIR,
                                                                           manifest_path=INPUTS / 'IMPLEMENTATION_MANIFEST.md')]
assert len(CHECKPOINTS) == 1 + 3 * S
MODE = 'CONFIRMATORY (S = 3)' if FROZEN['confirmatory'] else f'DESCRIPTIVE-ONLY reduced-S report (S = {S}): no confirmatory label'
GATE_PASSED = True
(STORE / 'final_test').mkdir(parents=True, exist_ok=True)
(STORE / 'final_test' / 'gate.json').write_text(json.dumps({'preregistration_sha256': PREREG_SHA256, 'freeze_manifest_sha256': FROZEN_SHA256,
    'implementation_manifest_sha256': MANIFEST_SHA256, 'baseline_dev_results_zip_sha256': FROZEN['baseline_dev_results_zip_sha256'],
    'S': S, 'confirmatory': FROZEN['confirmatory'],
    'descriptive_only': FROZEN['descriptive_only'], 'checkpoints': [[name, adapter] for name, adapter, _ in CHECKPOINTS],
    'passed_at': time.time()}, indent=1))
print('GATE PASSED:', MODE, [name for name, _, _ in CHECKPOINTS])


In [ ]:
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:  # CPU runtime: no nvidia-smi at all
    gpu = ''
print('GPU:', gpu or 'none')
assert 'T4' in gpu, 'Select Runtime → Change runtime type → T4 GPU'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements-phase2.txt')], check=True)
# A fresh interpreter sees the installed versions; this kernel never imports torch.
STACK = json.loads(subprocess.run([sys.executable, '-c', 'import sys; sys.path.insert(0, sys.argv[1]); import train_smoke as t, json; '
                                   'v = t.versions(); print(json.dumps({"versions": v, "mismatches": t.stack_mismatches(v)}))',
                                   str(PACKAGE / 't4_smoke')], capture_output=True, text=True, check=True).stdout)
print(json.dumps(STACK['versions'], indent=1))
STACK_MISMATCHES = STACK['mismatches']
print('D021 stack mismatches:', STACK_MISMATCHES or 'none')
# Amendment 2 item 3: only the exact §2 stack on a T4 can produce a study run; there is no override.
assert not STACK_MISMATCHES, 'D021 stack mismatch: no study run until a dated amendment (Amendment 2 item 3): ' + '; '.join(STACK_MISMATCHES)
# Tokenizer, chat template and config files at the pinned revision must match the implementation manifest.
HUB_CHECK = """
import hashlib, json, os, sys
from huggingface_hub import hf_hub_download
model = json.loads(sys.argv[1])
for name, expected in sorted(model['hub_files'].items()):
    path = hf_hub_download(model['id'], name, revision=model['revision'], token=os.environ.get('HF_TOKEN') or None)
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert actual == expected, f'{name} at the pinned revision has SHA-256 {actual}, manifest {expected}: refusing to run'
print('Pinned Hub files match the implementation manifest:', ', '.join(sorted(model['hub_files'])))
"""
print(subprocess.run([sys.executable, '-c', HUB_CHECK, json.dumps(MANIFEST['model'])], capture_output=True, text=True, check=True).stdout)


In [ ]:
assert GATE_PASSED, 'Gate did not pass'
for name, adapter, folder in CHECKPOINTS:  # sealed: phase2_eval prints completion/checksum/infrastructure only
    run_dir = STORE / 'final_test' / name
    args = ['evaluate', '--split', 'final_test', '--sealed', '--inputs', INPUTS, '--run-dir', run_dir,
            '--train-root', STORE / 'train', '--baseline-dir', BASELINE_DIR,
            '--implementation-manifest', INPUTS / 'IMPLEMENTATION_MANIFEST.md', *(['--recover-stale-lock'] if RECOVER_STALE_LOCK else [])]
    if adapter is not None:
        record = run_dir.parent / f'{name}_adapter_record.json'
        if not record.exists():
            record.write_text(json.dumps(adapter, indent=1))
        args += ['--adapter-dir', folder, '--adapter-record', record]
    step(run_dir.parent, 'phase2_eval.py', *args)
def sealed(name):
    path = STORE / 'final_test' / f'{name}_status.json'
    return json.loads(path.read_text()) if path.exists() else {'run': name, 'status': 'NOT STARTED'}
STATUS = [sealed(name) for name, _, _ in CHECKPOINTS]
for item in STATUS:
    print(json.dumps({k: item.get(k) for k in ('run', 'status', 'completed', 'expected', 'unresolved', 'attempts',
                                               'infrastructure_failures', 'results_zip_sha256')}))
assert all(item['status'] == 'COMPLETE' for item in STATUS), 'Incomplete: re-run all to resume (generated responses are kept, never retried)'
print('All', len(STATUS), 'FINAL_TEST runs complete.', MODE)


## Download
If a run is incomplete, re-run all first; then run this cell. The ZIP holds the sealed scored records; do not open them before every run is complete.


In [ ]:
zip_results(STORE / 'final_test', PACKAGE / 'Phase2_FinalTest_results.zip', 'final_test')
